## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from pint import Quantity

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.processing.figure_of_merit import gaussian
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers.get_midpoints_from_bins import get_midpoints_from_bins
from data_processing.processing.spectrum_unfolding import NDHistogram, unfold_spectrum, _nan_divide

### Functions

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


In [ ]:
# def relative_rmse(x: pd.Series | float, x_err: pd.Series | float, y: pd.Series | float, y_err: pd.Series | float) -> pd.Series | float:
def relative_rmse(values: list[tuple[pd.Series | float, pd.Series | float]]) -> pd.Series | float:
    rel_sq_values = [relative_square_error(x, x_err) for x, x_err in values]
    # rel_sq_x = relative_square_error(x, x_err)
    # rel_sq_y = relative_square_error(y, y_err)
    # rel_sq_sum = rel_sq_x + rel_sq_y
    rel_sq_sum = sum(rel_sq_values)
    if isinstance(rel_sq_sum, pd.Series):
        return rel_sq_sum.pow(1./2)
    else:
        return rel_sq_sum ** (1./2)


def relative_square_error(x: pd.Series | float, x_err: pd.Series | float) -> pd.Series | float:
    # divide x_err by x
    # square it
    # return
    rel_err = x_err / x
    if isinstance(rel_err, pd.Series):
        return rel_err.pow(2).fillna(0)
    else:
        return rel_err ** 2

## Data Loading

### Loading Params

In [ ]:
matrix_dir_name = "response_matrix_R4_mono"

In [ ]:
experiment_ids = ["ID-418"]

In [ ]:
exp_id = "ID-423"

phd_data_path = Path() / "pulse_height_distribution"
phd_input_path = phd_data_path / "input"
phd_output_path = phd_data_path / "output"
exp_data_filename = f"{exp_id}-n_spectrum.csv"
exp_data_path = phd_input_path / exp_data_filename

sim_data_filename = "highEstat-sig0.05MeV_A0.5_B0.01_C0.001.csv.npy"  # used in Fig 7b
sim_data_path = phd_input_path / sim_data_filename

In [ ]:
# calib_input = helpers.get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = (
    ExperimentDataKey.NEW_CALIBRATION
    if is_new_calibration
    else ExperimentDataKey.CAEN_CALIBRATION
)

In [ ]:
# detector_code = helpers.get_input_required(
#     """\
# Which detector was used?
# 1: Original detector (detector 1)
# 2: New detector (detector 2)
# """,
#     [Detector.ZERO, Detector.ONE],
#     lambda x: Detector(int(x)-1)
# )
detector_code = Detector.ZERO

In [ ]:
default_fit_input = 2  # changed to peak finder mode, approved by Fatima 2024-07-18
# fit_input = helpers.get_input_with_default(
#     """\
# Which bimodal fit type do you want to use?
# 1: Bounds based
# 2: Peak finder based (default)
# Press Enter for default
# """,
#     default_fit_input,
#     int
# )
fit_input = 2

fit_styles: dict[int, SliceFitStyle] = {
    1: "bounds",
    2: "peak_finder"
}
fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])

In [ ]:
# kind of window (Nasa, N distribution)
# load or generate
# specific settings for each condition to make namedtuple
# - generator settings (i.e. sigma, etc.)
# - file path prefix for loading
# done = False
strategy_factory = NeutronStrategyFactory()

# while not done:
#     window_input = helpers.get_input_with_default(
#         """\
# Which neutron classification window do you want to use?
# 1: NASA window (default)
# 2: Neutron distribution window
# Press Enter for default
# """,
#         1,
#         int
#     )
#     load_window_input = helpers.get_input_with_default(
#         """\
# Do you want to load the borders from the standard border file?
# [y/n, or press Enter for no]
# """,
#         "n",
#         str
#     )
#     done = True
#     will_load = load_window_input == "y"

#     try:
#         if window_input == 1:
#             if will_load:
#                 settings = get_nasa_loading_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "nasa", True, settings
#                 )
#             else:
#                 settings = get_nasa_generation_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "nasa", False, settings
#                 )
#                 pass
#         elif window_input == 2:
#             if will_load:
#                 settings = get_n_distro_loading_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "n_distro", True, settings
#                 )
#             else:
#                 settings = get_n_distro_generation_settings()
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "n_distro", False, settings
#                 )
#         else:
#             print("Invalid classification window type given, please try again")
#             done = False
#     except ValueError as err:
#         print("Problem found:")
#         print(err)
#         print("Please try again")
#         done = False

settings = NasaGenerationSettings(
    window_offset=0.2,
    sigma=5,
    lower_energy_bound=0.05,
    recalculate_lower_energy_bound=False
)
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings
)

In [ ]:
# bin_length = helpers.get_input_with_default(
#     "Enter bin length (in seconds), or press Enter for default (300 s)",
#     300,
#     int
# )
# bin_string = f"{bin_length}S"

In [ ]:
# bins_min = helpers.get_input_with_default(
#     "Enter minimum light output (in MeVee), or press Enter for default (0 MeVee)",
#     0,
#     float
# )
# bins_max = helpers.get_input_with_default(
#     "Enter maximum light output (in MeVee), or press Enter for default (1.2 MeVee)",
#     6,
#     float
# )
# bins_width = helpers.get_input_with_default(
#     "Enter light output bin width (in MeVee), or press Enter for default (0.02 MeVee)",
#     0.02,
#     float
# )
bins_min = 0
bins_max = 6
bins_width = 0.02

L_bins = np.arange(bins_min, bins_max + bins_width, bins_width).tolist()

### Loading

In [ ]:
figure5_data = {k: {} for k in ["A", "B", "C", "D"]}

In [ ]:
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}
experiment_neutron_data = make_strategy_for_experiments(experiment_neutron_data, factory_fn)

In [ ]:
filepath = Path("manuscript_plot_generators/CalibrationCurveData.csv")
df = pd.read_csv(filepath, dtype={"Isotope": "|S", "EnergyPeak": float, "LightOutput": float, "ADCchannel": int})
figure5_data["A"]["df"] = df
# df.dtypes

In [ ]:
R = load_neutron_response_matrix(
    Path(matrix_dir_name),
    min_L=bins_min,
    max_L=bins_max,
    L_bin_widths=bins_width
)
figure5_data["B"]["response_matrix"] = R
figure5_data["D"]["response_matrix"] = R

In [ ]:
exp_df = pd.read_csv(
    exp_data_path,
    usecols=[
        "Light output bin start (MeVee)",
        "Light output bin end (MeVee)",
        "Neutron counts"
    ]
)

figure5_data["C"]["exp_df"] = exp_df

In [ ]:
sim_L = np.load(sim_data_path)

figure5_data["C"]["sim_np"] = sim_L

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = recalibrate(unclassified_df, detector_code)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Data Processing

### Figure 5a Processing

In [ ]:
fig_5a_data = figure5_data["A"]

In [ ]:
df = fig_5a_data["df"]
cs137_df = df[df["Isotope"].str.contains("Cs137")]
co60_df = df[df["Isotope"].str.contains("Co60")]
na22_df = df[df["Isotope"].str.contains("Na22")]
eu152_df = df[df["Isotope"].str.contains("Eu152")]
isotope_dfs = {"Cs137": cs137_df, "Co60": co60_df, "Na22": na22_df, "Eu152": eu152_df}
fig_5a_data["isotope_data"] = isotope_dfs

In [ ]:
df = fig_5a_data["df"]
fit_params = np.polyfit(df["ADCchannel"], df["LightOutput"], 1)
# fit_params = np.polyfit(df["LightOutput"], df["ADCchannel"], 1)
fit_slope, fit_intercept = fit_params
print(fit_slope, fit_intercept)
fit_param_dict = {"slope": fit_slope, "intercept": fit_intercept}
fig_5a_data["fit_params"] = fit_param_dict

### Figure 5b Processing

In [ ]:
fig_5b_data = figure5_data["B"]

In [ ]:
R = fig_5b_data["response_matrix"]
L_mids, phi_mids = R.midpoints
L_widths = (L_mids[1:] - L_mids[:-1])
L_widths = np.insert(L_widths, (0,), L_widths[0])
phi_widths = phi_mids[1:] - phi_mids[:-1]
phi_widths = np.insert(phi_widths, (0,), phi_widths[0])

_x = phi_mids - (phi_widths / 2)
_y = L_mids - (L_widths / 2)
_xx, _yy = np.meshgrid(_x, _y)
x, y = np.ravel(_xx), np.ravel(_yy)

_xxw, _yyw = np.meshgrid(phi_widths, L_widths)
dx, dy = np.ravel(_xxw), np.ravel(_yyw)

# tops = [np.ravel(weight.counts) for weight in weights]
dz = np.ravel(R.counts)
with np.errstate(invalid="ignore", divide="ignore"):
    dz = np.nan_to_num(np.log10(dz), nan=np.nan, posinf=np.nan, neginf=np.nan)
    Rcounts = np.nan_to_num(np.log10(R.counts), nan=0, posinf=0, neginf=0)
z = np.zeros_like(dz)

# Zmax_list = [np.nanmax(top) for top in tops]
# Zmax = max(top)
# Zmin = 0

R_bar_plot_data = (x, y, z, dx, dy, dz)
R_surface_plot_data = (_xx, _yy, Rcounts)
fig_5b_data["bar_plot_data"] = R_bar_plot_data
fig_5b_data["surface_plot_data"] = R_surface_plot_data

In [ ]:
x, y, _ = fig_5b_data["surface_plot_data"]
R = fig_5b_data["response_matrix"]
z = R.counts
zero_idx = np.argmax(z == 0, axis=0)
col_idxs = range(z.shape[1])
x_baseline = x[zero_idx, col_idxs]
y_baseline = y[zero_idx, col_idxs]
z_baseline = np.zeros_like(x_baseline)
fig_5b_data["baseline_coords"] = (x_baseline.ravel(), y_baseline.ravel(), z_baseline.ravel())

In [ ]:
x, y, z = fig_5b_data["surface_plot_data"]
# R = fig_5b_data["response_matrix"]
# z = R.counts
expected_energy = 2.449  # closest in matrix to 2.45
e_idx = np.argmax(x[0] >= expected_energy)
e_response = z[:49, e_idx]
L_response = y[:49, e_idx]
phi_response = np.full_like(e_response, expected_energy)
fig_5b_data["exp_resp_coords"] = (phi_response.ravel(), L_response.ravel(), e_response.ravel())

### Figure 5c Processing

In [ ]:
fig_5c_data = figure5_data["C"]

In [ ]:
exp_df = fig_5c_data["exp_df"]
sim_np = fig_5c_data["sim_np"]

In [ ]:
exp_df['Count error'] = exp_df['Neutron counts'].pow(1./2)

exp_df = exp_df[exp_df["Light output bin start (MeVee)"] >= 0.05]

midpoints = helpers.get_midpoints_from_min_max_series(
    exp_df["Light output bin start (MeVee)"],
    exp_df["Light output bin end (MeVee)"],
    exp_df.index
)
exp_df["Light output (MeVee)"] = midpoints
fig_5c_data["exp_df"] = exp_df
# TODO fix SettingWithCopyWarning

In [ ]:
bins_lo = exp_df["Light output bin start (MeVee)"].to_numpy()
bins_hi = exp_df["Light output bin end (MeVee)"].to_numpy()
bins = np.append(bins_lo, bins_hi[-1])

binned_sim_L, *_ = np.histogram(sim_L, bins=bins)
bin_mids = (bins_hi + bins_lo) / 2

sim_df = pd.DataFrame({"Light output (MeVee)": bin_mids, "Count rate": binned_sim_L})
sim_df["Rate error"] = sim_df["Count rate"].pow(1./2)

fig_5c_data["sim_df"] = sim_df

#### Normalization

In [ ]:
counts = exp_df['Neutron counts']
errors = exp_df['Count error']

exp_max = counts.max()
# print(exp_max)
exp_max_idx = counts.idxmax()
exp_max_error = errors.loc[exp_max_idx]
rel_square_max_error = (exp_max_error / exp_max) ** 2

# norm_counts = counts / exp_max
norm_counts = counts
exp_df['Counts (normalized)'] = norm_counts

# rel_norm_errors = relative_rmse(counts, errors, exp_max, exp_max_error)
# norm_errors = rel_norm_errors * norm_counts
norm_errors = errors
exp_df['Error (normalized)'] = norm_errors

fig_5c_data["exp_df"] = exp_df
# TODO fix SettingWithCopyWarning

In [ ]:
counts = sim_df['Count rate']
errors = sim_df['Rate error']
sim_max = counts.max()
sim_max_idx = counts.idxmax()
sim_max_error = errors[sim_max_idx]
# print(sim_max)

sim_factor = exp_max / sim_max
norm_counts = counts * sim_factor
sim_df['Counts (normalized)'] = norm_counts

rel_norm_errors = relative_rmse(
    [(counts, errors),
     (exp_max, exp_max_error),
     (sim_max, sim_max_error)]
)
norm_errors = rel_norm_errors * norm_counts
# print(norm_errors)
sim_df['Error (normalized)'] = norm_errors

fig_5c_data["sim_df"] = sim_df

### Figure 5d Processing

In [ ]:
fig_5d_data = figure5_data["D"]
R = fig_5d_data["response_matrix"]

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 5e-3
# overall_settings['scan_idx'] = f"({start_scan_idx}, {end_scan_idx})"
# overall_settings['energy_width'] = energy_width

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    if fit_style == "bounds":
        # Default
        default_bounds: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
        )

        bounds_a: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
        )

        bounds_b: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
        )

        # Ranged Example
        bounds = [
            ((0, 60), bounds_a),
        ]
    else:
        default_bounds = None
        bounds = None

    df, df_err = scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style=fit_style,
        default_bounds=default_bounds,
        bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = find_failed_slices(df, exp_id, nan_total_threshold=10)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    helpers.stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Get experiment start time
for exp_name, data_dict in experiment_neutron_data.items():
    exp_root = get_exp_root(exp_name)
    with open(exp_root / 'exp_info.toml') as exp_info:
        exp_start_line = [line for line in exp_info if "exp_start" in line][0]
    exp_start_text = exp_start_line.replace("exp_start = ", "").strip()
    exp_start = datetime.fromisoformat(exp_start_text).astimezone(timezone.utc)
    data_dict[ExperimentDataKey.START_TIME] = exp_start

In [ ]:
# Get timetag as clock time
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    exp_start = data_dict[ExperimentDataKey.START_TIME]

    psd_report = calculate_event_time(psd_report, exp_start)

    data_dict[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Separate neutron and gamma events
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]

    n_classify_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    neutrons_only = psd_report.query(n_classify_col_name).copy()
    gamma_only = psd_report.query(f"~{n_classify_col_name}").copy()
    data_dict[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    data_dict[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
# Get total experiment time
for exp_name, data_dict in experiment_neutron_data.items():
    neutrons_only = data_dict[ExperimentDataKey.NEUTRONS_ONLY]
    max_timetag = neutrons_only["TIMETAG"].max()
    data_dict["max_seconds"] = max_timetag * 1E-12

#### Unfold Neutron Spectrum

In [ ]:
# make PSD histogram
for exp_id, exp_data in experiment_neutron_data.items():
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    max_seconds = exp_data["max_seconds"]
    neutron_energies = neutrons_only[calibrated_energy_column.value]
    energy_bins = np.arange(start=bins_min, stop=bins_max + bins_width, step=bins_width)
    
    # TODO generate neutron PHD histogram
    Z_n, *_ = np.histogram(neutron_energies, bins=energy_bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
        "neutron": {"standard": Z_n, "standard_edges": energy_bins}
    }

In [ ]:
# Normalization

# # By E slice max
R_max = R.counts.max(axis=0, keepdims=True)

In [ ]:
# Normalization
for exp_id, exp_data in experiment_neutron_data.items():
    Z_n = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]["standard"]
    Z_n = Z_n.reshape(-1, 1)

    Z_norm = Z_n.copy()
    # print(Z_norm.shape)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]["normalized"] = Z_norm

In [ ]:
R_L_mids, R_E_mids = R.midpoints
for exp_id, exp_data in experiment_neutron_data.items():
    PHD = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]["normalized"]
    PHD_bins = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]["standard_edges"]
    PHD_mids = get_midpoints_from_bins(PHD_bins)
    if PHD_mids.shape[0] != R_L_mids.shape[0] or not all(np.isclose(PHD_mids, R_L_mids)):
        raise ValueError("Bin mismatch!")
    reduced_E_mids = np.array([R_E_mids.mean()])
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]["standard_histogram"] = NDHistogram(PHD, [R_L_mids, reduced_E_mids])

In [ ]:
tol = 0.01

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    N = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]["standard_histogram"]
    phi, unfold_info = unfold_spectrum(R, N, full_info=True, tolerance=tol, L_cut=0.05)
    exp_data["neutron_spectrum"] = phi
    exp_data["unfolding_info"] = unfold_info

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    phi = exp_data["neutron_spectrum"]
    _, x = phi.midpoints
    y = phi.counts.reshape(-1)
    y = np.nan_to_num(y)
    fit_params, fit_cov = curve_fit(gaussian, x, y)
    print(fit_params)
    print(np.sqrt(np.diag(fit_cov)))
    exp_data["spectrum_fit"] = fit_params

In [ ]:
base_errorbars = [(1.798, (0.31000000000000005, 0.6200000000000001)),
 (1.86, (0.31000000000000005, 0.558)),
 (1.922, (0.3719999999999999, 0.558)),
 (1.984, (0.3719999999999999, 0.496)),
 (2.046, (0.3719999999999999, 0.496)),
 (2.108, (0.3720000000000001, 0.496)),
 (2.17, (0.43399999999999994, 0.43400000000000016)),
 (2.232, (0.3720000000000001, 0.3719999999999999)),
 (2.294, (0.3720000000000001, 0.3719999999999999)),
 (2.418, (0.43400000000000016, 0.31000000000000005)),
 (2.48, (0.496, 0.31000000000000005)),
 (2.604, (0.3719999999999999, 0.4339999999999997)),
 (2.666, (0.4339999999999997, 0.3719999999999999)),
 (2.79, (0.496, 0.3719999999999999)),
 (2.852, (0.3719999999999999, 0.43400000000000016)),
 (2.914, (0.43400000000000016, 0.3719999999999999)),
 (2.976, (0.496, 0.18599999999999994)),
 (3.038, (0.496, 0.1860000000000004)),
 (3.1, (0.5580000000000003, 0.18599999999999994)),
 (3.162, (0.496, 0.18599999999999994)),
 (3.224, (0.5580000000000003, 0.18599999999999994)),
 (3.286, (0.5579999999999998, 0.18599999999999994))]

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    phi = exp_data["neutron_spectrum"]
    phi_mids = phi.midpoints[1]
    errorbars = []
    for mid in phi_mids:
        isclose = [np.isclose(x, mid) for x, _ in base_errorbars]
        true_idx = [i for i, x in enumerate(isclose) if x]
        if len(true_idx) == 0:
            errorbars.append((np.nan, np.nan))
            continue
        idx = true_idx[0]
        _, errorbar = base_errorbars[idx]
        errorbars.append(errorbar)
    errorbars = list(zip(*errorbars))
    exp_data["spectrum_xerr"] = errorbars

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    N = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]["standard_histogram"]
    phi = exp_data["neutron_spectrum"]
    xerr = exp_data["spectrum_xerr"]

    phi_flat = phi.counts.reshape(-1)
    xerr_left, _ = xerr

    sigma_counts = np.sqrt(N.counts)
    sigma = NDHistogram(sigma_counts, N.midpoints)
    N_yerr_plus = NDHistogram(N.counts + 3 * sigma_counts, N.midpoints)
    N_yerr_minus = NDHistogram(N.counts - 3 * sigma_counts, N.midpoints)

    phi_yerr_plus, _ = unfold_spectrum(
        R,
        N_yerr_plus,
        L_cut=0.05,
        # tolerance=0.0000001,
        max_iterations=1000,
        sigma=sigma
    )
    phi_yerr_minus, _ = unfold_spectrum(
        R,
        N_yerr_minus,
        L_cut=0.05,
        # tolerance=0.0000001,
        max_iterations=1000,
        sigma=sigma
    )

    phi_yerr_plus_flat = phi_yerr_plus.counts.reshape(-1)
    phi_yerr_minus_flat = phi_yerr_minus.counts.reshape(-1)
    
    plus_gt_phi = (phi_yerr_plus_flat >= phi_flat) | (np.isnan(phi_flat))
    phi_yerr_larger = phi_yerr_plus_flat.copy()
    phi_yerr_larger[~plus_gt_phi] = phi_yerr_minus_flat[~plus_gt_phi]
    
    minus_lt_phi = (phi_yerr_minus_flat <= phi_flat) | (np.isnan(phi_flat))
    phi_yerr_smaller = phi_yerr_minus_flat.copy()
    phi_yerr_smaller[~minus_lt_phi] = phi_yerr_plus_flat[~minus_lt_phi]

    yerr_up = np.nan_to_num(phi_yerr_larger - phi_flat, nan=np.nan)
    yerr_up[yerr_up < 0] = 0
    yerr_up[np.isnan(xerr_left)] = np.nan
    
    yerr_down = np.nan_to_num(phi_flat - phi_yerr_smaller, nan=np.nan)
    yerr_down[yerr_down < 0] = 0
    yerr_down[np.isnan(xerr_left)] = np.nan

    yerr = [yerr_down, yerr_up]
    exp_data["spectrum_yerr"] = yerr

In [ ]:
exp_data = experiment_neutron_data["ID-418"]
fig_5d_data["neutron_spectrum"] = exp_data["neutron_spectrum"]
fig_5d_data["spectrum_xerr"] = exp_data["spectrum_xerr"]
fig_5d_data["spectrum_yerr"] = exp_data["spectrum_yerr"]
fig_5d_data["spectrum_fit"] = exp_data["spectrum_fit"]

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"

In [ ]:
import matplotlib as mpl

vaporwave_colors = [
    [255, 255, 255],
    [128, 69, 229],
    [75,127,255],
    [0,255,255],
    [255,186,129],
    [255,209,86],
    [252,120,183]
]
vaporwave_colors = [[value/255 for value in color] for color in vaporwave_colors]
vaporwave = mpl.colors.LinearSegmentedColormap.from_list("vaporwave", vaporwave_colors, N=256)
# vaporwave = vaporwave.resampled(256)
vaporwave

In [ ]:
def plot_figure_5a(ax: mpl.axes.Axes):
    fig_5a_data = figure5_data["A"]
    df = fig_5a_data["df"]
    isotope_data = fig_5a_data["isotope_data"]
    fig_params_data = fig_5a_data["fit_params"]
    cs137_df = isotope_data["Cs137"]
    co60_df = isotope_data["Co60"]
    na22_df = isotope_data["Na22"]
    eu152_df = isotope_data["Eu152"]
    fit_slope = fig_params_data["slope"]
    fit_intercept = fig_params_data["intercept"]
    for isotope_df, isotope_label, marker_color in zip(
        [cs137_df, co60_df, na22_df, eu152_df],
        [r"${}^{137}Cs$", r"${}^{60}Co$", r"${}^{22}Na$", r"${}^{152}Eu$"],
        [bg_blue, bg_red, bg_grey, bg_green]
    ):
        iso_adc = isotope_df["ADCchannel"]
        iso_light = isotope_df["LightOutput"]
        ax.plot(iso_adc, iso_light, "o", ms=10, label=isotope_label, color=marker_color)

    max_adc = df["ADCchannel"].max()
    fit_adc = np.linspace(0, max_adc, 100)
    fit_light = fit_slope * fit_adc + fit_intercept
    ax.plot(fit_adc, fit_light, "-", label="Fit", lw=3)

    ax.tick_params(labelsize=fontsize)
    ax.set_xlabel("ADC channel", fontsize=fontsize)
    ax.set_ylabel("Light output (keVee)", fontsize=fontsize)

In [ ]:
def plot_figure_5b_bar(ax: mpl.axes.Axes):
    fig_5b_data = figure5_data["B"]
    x, y, z, dx, dy, dz = fig_5b_data["bar_plot_data"]
    # x, y, z = fig_5b_data["surface_plot_data"]
    Zmin = 0
    Zmax = max(dz)
    norm = mpl.colors.Normalize(vmin=Zmin, vmax=Zmax)
    sc = mpl.cm.ScalarMappable(cmap=vaporwave, norm=norm)
    sc.set_array([])
    
    # weights_colors = [vaporwave(norm(top)) for top in tops]
    R_color = vaporwave(norm(dz))
    ax.view_init(elev=30, azim=45)

    # artists = []
    # _frame = 0
    # for i in range(showevery_weight-1, len(tops), showevery_weight):
    #     print(f"{_frame % 10}", end="")
    #     _frame += 1
    #     top = tops[i]
    #     color = weights_colors[i]
    nan_mask = ~np.isnan(dz)
    # masked_args = [param[nan_mask] for param in [x, y, bottom, xw, yw, top]]
    masked_args = [param[nan_mask] for param in [x, y, z, dx, dy, dz]]
    masked_color = R_color[nan_mask]
    # W_plot = ax.bar3d(x, y, bottom, xw, yw, top, color=color)
    ax.bar3d(*masked_args, color=masked_color)


def plot_figure_5b_surface(ax: mpl.axes.Axes):
    fig_5b_data = figure5_data["B"]
    x, y, z = fig_5b_data["surface_plot_data"]
    base_x, base_y, base_z = fig_5b_data["baseline_coords"]
    exp_x, exp_y, exp_z = fig_5b_data["exp_resp_coords"]
    Zmin = 0
    Zmax = z.max()
    norm = mpl.colors.Normalize(vmin=Zmin, vmax=Zmax)
    # sc = mpl.cm.ScalarMappable(cmap=vaporwave, norm=norm)
    # sc.set_array([])
    
    # weights_colors = [vaporwave(norm(top)) for top in tops]
    # R_color = vaporwave(norm(z))
    # print(R_color)
    ax.view_init(elev=30, azim=45)
    # ax.view_init(elev=90, azim=0)

    # artists = []
    # _frame = 0
    # for i in range(showevery_weight-1, len(tops), showevery_weight):
    #     print(f"{_frame % 10}", end="")
    #     _frame += 1
    #     top = tops[i]
    #     color = weights_colors[i]
    # nan_mask = ~np.isnan(z)
    # # masked_args = [param[nan_mask] for param in [x, y, bottom, xw, yw, top]]
    # masked_args = [param[nan_mask] for param in [x, y, z]]
    # masked_color = R_color[nan_mask]
    # W_plot = ax.bar3d(x, y, bottom, xw, yw, top, color=color)
    # ax.plot_surface(*masked_args, color=masked_color)
    ax.plot_surface(x, y, z, cmap="viridis", norm=norm, vmin=Zmin, vmax=Zmax, shade=False, axlim_clip=True)
    # ax.plot_wireframe(x, y, z, alpha=0.5, rstride=10, cstride=5, axlim_clip=True, zorder=1)
    # ax.plot(base_x, base_y, base_z, "r-", zorder=2, lw=3)
    ax.plot(exp_x, exp_y, exp_z, "w-", zorder=20, lw=3)
    
    ax.set_ylim(0, 4.0)
    ax.set_xlabel("Neutron energy (MeV)", fontsize=fontsize)
    ax.set_ylabel("Light output (MeVee)", fontsize=fontsize)
    ax.zaxis.set_rotate_label(False)
    ax.set_zlabel(r"$\mathrm{Log_{10}}$ counts", fontsize=fontsize, rotation=90)
    # ax.yaxis.set_major_formatter(lambda y, _: f"{int(y*1000)}")
    ax.xaxis.set_major_formatter("{x:.1f}")
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(1))
    ax.yaxis.set_major_formatter("{x:.1f}")
    ax.zaxis.set_major_formatter(lambda z, _: str(z) if z != 0 else "")
    ax.tick_params(labelsize=fontsize, pad=0)
    ax.tick_params("z", pad=-2)
    ax.xaxis.labelpad = 0.5 * fontsize
    ax.yaxis.labelpad = 0.7 * fontsize
    ax.zaxis.labelpad = 0.8 * fontsize
    x_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in x_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("center")
    y_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in y_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("center")
    z_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in z_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("center")
    ax.set_box_aspect(None, zoom=0.90)

In [ ]:
def plot_figure_5c(ax: mpl.axes.Axes):
    fig_5c_data = figure5_data["C"]
    sim_df = fig_5c_data["sim_df"]
    exp_df = fig_5c_data["exp_df"]
    # print(exp_df.head())
    # print(sim_df.head())

    sim_x = sim_df["Light output (MeVee)"].astype(float) * 1000
    sim_y = sim_df["Counts (normalized)"].astype(float)
    sim_error = sim_df["Error (normalized)"].astype(float)
    # sim_raw_y = sim_df["Count rate"].astype(float)
    # sim_raw_error = sim_df["Rate error"].astype(float)

    exp_x = exp_df["Light output (MeVee)"] * 1000
    exp_y = exp_df["Counts (normalized)"]
    exp_error = exp_df["Error (normalized)"]
    # exp_raw_y = exp_df["Neutron counts"]
    # exp_raw_y_error = exp_df["Count error"]

    ax.errorbar(
        sim_x,
        sim_y,
        yerr=sim_error,
        ls='-',
        marker='',
        ms=10,
        capsize=4,
        lw=3
    )
    ax.errorbar(
        exp_x,
        exp_y,
        yerr=exp_error,
        ls='--',
        marker='',
        ms=10,
        capsize=4,
        lw=3
    )
    ax.set_xlim(-20, 1400)
    ax.set_ylim(1, 100000)
    # axs.set_ylim(-5,180)
    ax.set_ylabel('Normalized counts', fontsize=fontsize)
    ax.set_yscale("log")
    ax.set_xlabel('Light output (keVee)', fontsize=fontsize)
    # axs.axhline(64, color = 'black', ls = "--", alpha = 0.7)
    # axs.axhline(45, color = 'black', ls = "--", alpha = 0.7)
    ax.tick_params(axis="x", labelsize=fontsize)
    ax.tick_params(axis="y", labelsize=fontsize)

In [ ]:
def plot_figure_5d(ax: mpl.axes.Axes):
    fig_5d_data = figure5_data["D"]
    phi = fig_5d_data["neutron_spectrum"]
    xerr = fig_5d_data["spectrum_xerr"]
    yerr = fig_5d_data["spectrum_yerr"]
    fit_params = fig_5d_data["spectrum_fit"]

    _, phi_mids = phi.midpoints
    phi_counts = phi.counts.reshape(-1)
    ax.errorbar(
        phi_mids,
        phi_counts,
        fmt="o",
        xerr=xerr,
        yerr=yerr,
        # marker="o",
        markersize=8,
        markerfacecolor=bg_blue,
        markeredgecolor=None,
        # lw=0,
        capsize=8,
        ecolor="black",
        # color="#ff7f0e",
        # lw=3
    )

    # fit_x = np.linspace(phi_mids[0], phi_mids[-1], 100)
    # fit_y = gaussian(fit_x, *fit_params)
    # ax.plot(
    #     fit_x,
    #     fit_y,
    #     "-",
    #     lw=3,
    #     color = bg_bluegrey
    # )

    ax.set_ylim(0, 3.5)
    ax.set_xlim(0, 6)
    ax.set_xlabel("Neutron energy (MeV)", fontsize=fontsize)
    ax.set_ylabel("Counts [arb. units]", fontsize=fontsize)
    ax.yaxis.set_major_formatter(lambda x, pos: "" if pos == 0 else f"{x:.1f}")
    ax.tick_params(labelsize=fontsize)

In [ ]:
dl_folder = Path.home() / "Downloads"

In [ ]:
# # mosaic = """
# # .AA.
# # BB..
# # BBCC
# # BB..
# # .DD.
# # """
# mosaic = """
# .AA.
# BB..
# BBCC
# BB..
# """
# fig, ax_dict = plt.subplot_mosaic(
#     mosaic,
#     # figsize=(35, 28),
#     figsize=(35, 26),
#     dpi=600,
#     width_ratios=[9, 10, 7, 9],
#     # height_ratios=[8, 2, 8, 2, 8],
#     height_ratios=[8, 1, 8, 1],
#     # subplot_kw={"xmargin": 0, "ymargin": 0},
#     per_subplot_kw={"B": {"projection": "3d", "computed_zorder": False}},
#     # layout="constrained"
# )

# plot_figure_5a(ax_dict["A"])
# # plot_figure_5b_surface(ax_dict["B"])
# # plot_figure_5c(ax_dict["C"])
# # plot_figure_5d(ax_dict["D"])

# # fig.text(0.52, 0.65, "Simulated detector response to 2.45 MeV monoenergetic neutron", fontsize=fontsize+8, ha="center", va="baseline")
# # fig.text(0.8, 0.62, "Experimental neutron response", fontsize=fontsize+8, ha="center", va="baseline")

# # start_a = (0.400, 0.645)
# # start_b = (0.650, 0.645)
# # start_c = (0.800, 0.615)
# # end_a = (0.333, 0.570)
# # end_b = (0.585, 0.577)
# # end_c = (0.730, 0.545)
# # delta_a = tuple([end - start for end, start in zip(end_a, start_a)])
# # delta_b = tuple([end - start for end, start in zip(end_b, start_b)])
# # delta_c = tuple([end - start for end, start in zip(end_c, start_c)])
# # # arrow_params = {
# # #     "length_includes_head": True,
# # #     # "width": 0.002
# # #     "head_width": 0.01,
# # #     "overhang": 0.4,
# # #     "ec": None,
# # #     "fc": "black"
# # # }
# # arrow_params = {
# #     "fc": "black",
# #     "lw": 3,
# #     "mutation_scale": 50,
# #     "arrowstyle": "-|>"
# # }
# # # arrow_a = mpl.patches.FancyArrow(*start_a, *delta_a, **arrow_params)
# # # arrow_b = mpl.patches.FancyArrow(*start_b, *delta_b, **arrow_params)
# # # arrow_c = mpl.patches.FancyArrow(*start_c, *delta_c, **arrow_params)
# # arrow_a = mpl.patches.FancyArrowPatch(start_a, end_a, **arrow_params)
# # arrow_b = mpl.patches.FancyArrowPatch(start_b, end_b, **arrow_params)
# # arrow_c = mpl.patches.FancyArrowPatch(start_c, end_c, **arrow_params)
# # fig.add_artist(arrow_a)
# # fig.add_artist(arrow_b)
# # fig.add_artist(arrow_c)

# # plt.tight_layout(pad=1.01)
# fig.savefig(dl_folder / "fig5.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
margin = 1.0 / 8

In [ ]:
figsize = (8.5, 6)
multiplier = 1
figsize = [size * multiplier for size in figsize]
lmargin = 0
rmargin = 0
tmargin = margin
bmargin = margin / 2
fig_margins = (lmargin + rmargin, tmargin + bmargin)
figsize_adjust = [size - margin for size, margin in zip(figsize, fig_margins)]

fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_adjust,
    dpi=600,
    # frameon=False,
    layout="constrained"
)
plot_figure_5a(ax)
fig.savefig(dl_folder / "fig5a.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
figsize = (8.5, 7)
multiplier = 1.5
figsize = [size * multiplier for size in figsize]
lmargin = margin
rmargin = margin / 2
tmargin = margin / 2
bmargin = margin / 2
fig_margins = (lmargin + rmargin, tmargin + bmargin)
figsize_adjust = [size - margin for size, margin in zip(figsize, fig_margins)]

fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_adjust,
    dpi=600,
    # frameon=False,
    subplot_kw={"projection": "3d"},
    layout="constrained"
)
plot_figure_5b_surface(ax)
ax.set_box_aspect(None, zoom=0.8)
fig.savefig(dl_folder / "fig5b1.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
figsize = (8.5, 7)
multiplier = 1
figsize = [size * multiplier for size in figsize]
lmargin = margin / 2
rmargin = margin
tmargin = margin / 2
bmargin = margin / 2
fig_margins = (lmargin + rmargin, tmargin + bmargin)
figsize_adjust = [size - margin for size, margin in zip(figsize, fig_margins)]

fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_adjust,
    dpi=600,
    # frameon=False,
    layout="constrained"
)
plot_figure_5c(ax)
fig.savefig(dl_folder / "fig5b2.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
figsize = (8.5, 7)
multiplier = 1
figsize = [size * multiplier for size in figsize]
lmargin = 0
rmargin = 0
tmargin = margin / 2
bmargin = margin
fig_margins = (lmargin + rmargin, tmargin + bmargin)
figsize_adjust = [size - margin for size, margin in zip(figsize, fig_margins)]

fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_adjust,
    dpi=600,
    # frameon=False,
    layout="constrained"
)
plot_figure_5d(ax)
fig.savefig(dl_folder / "fig5c.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()